[← Back to 08a - Changing Data, Part 1](<../08a-changing-data-part-1.ipynb>) · [Course Overview](<../00-course-overview.ipynb>)

# 08a - Changing Data, Part 1: Solutions

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/samietex/SQL_Tutorial/blob/main/solutions/08a-changing-data-part-1-solutions.ipynb)

Try each exercise in the lesson notebook first. If your query differs from the one here but returns the same rows, it may well be just as correct, and seeing why is a useful exercise in itself.

In [1]:
# Run this cell first. It downloads the small helper that lets the cells below run SQL,
# fetching a fresh copy every time so you always have the latest version.
import os, sys, tempfile, urllib.request
helper_dir = os.path.join(tempfile.gettempdir(), 'sql_tutorial_helper')
os.makedirs(helper_dir, exist_ok=True)
urllib.request.urlretrieve(
    'https://raw.githubusercontent.com/samietex/SQL_Tutorial/assets/sql_tutorial.py',
    os.path.join(helper_dir, 'sql_tutorial.py'),
)
sys.path.insert(0, helper_dir)
%reload_ext sql_tutorial

Ready. %%sql runs SQLite 3.50.4 in memory, %%tsql runs SQL Server.


In [2]:
%%sql
CREATE TABLE customers (
    customer_id INTEGER PRIMARY KEY,
    customer_name TEXT NOT NULL,
    credit_limit INTEGER NOT NULL
);
CREATE TABLE orders (
    order_id INTEGER PRIMARY KEY,
    customer_id INTEGER NOT NULL,
    amount INTEGER NOT NULL
);
INSERT INTO customers VALUES
    (1, 'Ada', 100),
    (2, 'Bola', 200),
    (3, 'Chidi', 300),
    (4, 'Dami', 200);
INSERT INTO orders VALUES
    (101, 1, 40),
    (102, 1, 60),
    (103, 2, 80),
    (104, 2, 90),
    (105, 3, 140);

4 rows inserted
5 rows inserted


The lesson notebook raised Bola's limit to 250 in section 4, and exercise 2 depends on that, so do the same here:

In [3]:
%%sql
UPDATE customers
SET credit_limit = credit_limit + 50
WHERE customer_id = 2;

1 row updated


## 1. Three statements, one table

In [4]:
%%sql
BEGIN;

INSERT INTO orders (
    order_id,
    customer_id,
    amount
)
VALUES (106, 4, 50);

UPDATE orders
SET amount = amount + 10
WHERE customer_id = 1;

DELETE FROM orders
WHERE amount < 50;

SELECT
    order_id,
    customer_id,
    amount
FROM orders
ORDER BY
    order_id;

ROLLBACK;

1 row inserted
2 rows updated
0 rows deleted


order_id,customer_id,amount
101,1,50
102,1,70
103,2,80
104,2,90
105,3,140
106,4,50


The `DELETE` removes nothing. The tempting answer is that order 101 goes, because it started at 40. But each statement sees the table the one before it left behind, and by the time the `DELETE` runs, the `UPDATE` has already raised 101 to 50, which isn't less than 50. Dami's new order 106 is exactly 50 too, so it stays. Swap the `UPDATE` and the `DELETE`, and order 101 is deleted while it's still 40. Order matters for writes in a way it never did for a single `SELECT`, and the `ROLLBACK` gives Dami back her clean record of zero orders.

## 2. A raise for customers who've ordered

In [5]:
%%sql
BEGIN;

UPDATE customers
SET credit_limit = credit_limit + credit_limit / 10
WHERE customer_id IN (SELECT customer_id FROM orders);

SELECT
    customer_name,
    credit_limit
FROM customers
ORDER BY
    customer_id;

ROLLBACK;

3 rows updated


customer_name,credit_limit
Ada,110
Bola,275
Chidi,330
Dami,200


Three rows updated: Ada to 110, Bola to 275, Chidi to 330, and Dami still at 200. Two details are worth noticing. The subquery decides who's affected, the same "check the `WHERE` first" principle, since you could run `SELECT customer_id FROM orders` on its own to see the list. And `credit_limit + credit_limit / 10` stays in whole numbers, because both sides are integers; `credit_limit * 1.1` would store values like 110.00000000000001, because 1.1 can't be represented exactly as a binary fraction.

## 3. Keep a copy before you change anything

In [6]:
%%sql
CREATE TABLE orders_backup AS
SELECT *
FROM orders;

In [7]:
%%sql
BEGIN;

DELETE FROM orders
WHERE amount > 100;

SELECT
    (SELECT COUNT(*) FROM orders) AS orders_now,
    (SELECT COUNT(*) FROM orders_backup) AS in_backup;

ROLLBACK;

1 row deleted


orders_now,in_backup
4,5


One order deleted (Chidi's 140), four left, and five safe in the backup. `CREATE TABLE ... AS SELECT` copies the rows but not the rules: the backup has no primary key or `NOT NULL` constraints, which is fine for a safety copy and worth knowing before you treat one as a real table. On SQL Server the same idea is `SELECT ... INTO`, as in section 9.

## 4. Stretch: lower the price of colorless products (AdventureWorks)

This answer runs on SQL Server, so set up AdventureWorks first. In Colab that takes a few minutes the first time, and on your own machine it only checks the connection.

In [8]:
# Run this once before any %%tsql cell.
# In Google Colab it installs SQL Server and AdventureWorks into this session (about 3 to 5
# minutes the first time, and again in each new session, because Colab starts fresh).
# On your own machine it just checks that your SQL Server is reachable.
%setup_adventureworks

AdventureWorks2022 is ready, so every %%tsql cell in this notebook will run.


In [9]:
%%tsql
SELECT
    ProductID,
    Name,
    Color,
    ListPrice
INTO #my_prices
FROM Production.Product
WHERE ListPrice > 0;

304 rows copied


In [10]:
%%tsql
BEGIN TRANSACTION;

UPDATE #my_prices
SET ListPrice = ListPrice * 0.9
WHERE Color IS NULL;

SELECT TOP 5
    Name,
    ListPrice
FROM #my_prices
WHERE Color IS NULL
ORDER BY
    ProductID;

ROLLBACK;

59 rows updated


Name,ListPrice
LL Mountain Seat Assembly,120.0060
ML Mountain Seat Assembly,132.4260
HL Mountain Seat Assembly,177.2280
LL Road Seat Assembly,120.0060
ML Road Seat Assembly,132.4260


59 rows updated, out of 304 priced products. `WHERE Color = NULL` would have updated nothing at all, with no error, the same trap from `02` in a place where it's more costly, because a write that silently changes nothing looks exactly like a write that worked.

---

[← Back to 08a - Changing Data, Part 1](<../08a-changing-data-part-1.ipynb>) · [Course Overview](<../00-course-overview.ipynb>)

*SQL_Tutorial* is written and maintained by Samuel Shaibu as part of *All About Data & More*. Licensed under [MIT](../LICENSE).